# OpenLineage и Marquez: происхождение данных


Показать жизненный цикл запуска, пространства имён, описание полей и безопасную проверку API.

Нужен `SparkSessionContext`; для проверки Marquez задайте `MARQUEZ_URL`. Без него соответствующая ячейка сообщает о пропуске. Примеры Spark требуют доступных таблиц.

Ограничения: Когда нужно ответить, откуда взялся показатель и какой запуск его вычислил. Не помещайте секреты в описание событий; точность связей полей зависит от корректного контракта имен.


## Основные понятия
OpenLineage описывает запуск, наборы входа и выхода и связи полей. Marquez принимает события и показывает граф.


In [ ]:
from lakehouse_spark import SparkSessionContext

import os
import requests

## Примеры


**1. Жизненный цикл**

Оформите чтение как запуск OpenLineage без записи новой таблицы.



In [ ]:
with SparkSessionContext("example-lineage-lifecycle") as spark:
    with spark.lineage(
        "example_summary", 
        inputs=["raw.pg__matches"], 
        description="Чтение длительности матчей"
    ):
        spark.sql("""
            select radiant_win, count(*) as matches 
            from lakehouse.raw.pg__matches 
            group by radiant_win 
            limit 10
        """).show()

**2. Пространства имён**

Покажите явные пространства имён для входного и вычисляемого набора.



In [ ]:
namespaces = {
    "input": "lakehouse_raw", 
    "derived": "jh_mart"

}
print(namespaces)

**3. Связи полей**

Опишите, из каких полей получены ограниченные вычисляемые поля.



In [ ]:
column_lineage = {
    "duration_minutes": ["raw.pg__matches.duration"],
    "total_kills": ["raw.pg__matches.radiant_score", "raw.pg__matches.dire_score"],
}

print(column_lineage)

**4. Проверка API**

Проверьте доступность Marquez, не раскрывая адрес с секретами.



In [ ]:
url = os.getenv("MARQUEZ_URL")
if not url:
    print("Пример пропущен: MARQUEZ_URL не задан")
else:
    response = requests.get(f"{url}/api/v1/namespaces", timeout=10)
    response.raise_for_status()
    
    print("Пространств:", len(response.json().get("namespaces", [])))

**5. Ошибка запуска**

Сделайте диагностический запуск с явным обработчиком исключения.



In [ ]:
try:
    with SparkSessionContext("example-lineage-failure") as spark:
        with spark.lineage("example_failure", inputs=["raw.pg__matches"]):
            raise RuntimeError("Учебная ошибка: проверка события FAIL")
except RuntimeError as error:
    print(type(error).__name__, str(error))